# Prompt Experiments with the Galileo Python SDK

This notebook keeps the dataset and experiment calls inline so you can see how Galileo datasets, prompt templates, and experiment runs are assembled step by step.


In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')


In [ ]:
from galileo import GalileoMetrics, galileo_context
from galileo.config import GalileoPythonConfig
from galileo.datasets import create_dataset, delete_dataset, get_dataset
from galileo.experiments import PromptRunSettings, PromptTemplate, run_experiment
from galileo.log_streams import create_log_stream, enable_metrics
from galileo.projects import delete_project

PROJECT_NAME = 'GalileoEval_S4_Experiments'
LOG_STREAM_NAME = 'experiment-lab'
DATASET_NAME = 'prompt-lab-dataset'
dataset_id = None


## 1. Initialize Galileo

Initialize the Galileo context, make sure the log stream exists, and print the console links if Galileo returns IDs.


In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

try:
    create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)
except Exception:
    pass

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    print('Galileo context initialized')


## 2. Create a Dataset


In [ ]:
ds = create_dataset(
    name=DATASET_NAME,
    content=[
        {'input': 'Explain photosynthesis in one sentence.', 'expected_output': 'Photosynthesis is the process by which plants convert sunlight, water, and CO2 into glucose and oxygen.'},
        {'input': 'What causes rain?', 'expected_output': 'Rain occurs when water vapor condenses into droplets that fall when heavy enough.'},
        {'input': 'Why is the sky blue?', 'expected_output': 'The sky appears blue because shorter blue wavelengths scatter more than other colors.'},
        {'input': 'How do vaccines work?', 'expected_output': 'Vaccines train the immune system to recognize and fight pathogens.'},
    ],
    project_name=PROJECT_NAME,
)
dataset_id = ds.id
str(dataset_id)


## 3. Add More Dataset Rows


In [ ]:
ds = get_dataset(id=str(dataset_id))
ds.add_rows([
    {'input': 'What is gravity?', 'expected_output': 'Gravity is the force of attraction between objects with mass.'},
    {'input': 'How does WiFi work?', 'expected_output': 'WiFi uses radio waves to transmit data wirelessly between devices and a router.'},
])
'Added 2 rows'


## 4. Run a Prompt Template Experiment


In [ ]:
prompt = PromptTemplate(
    messages=[
        {'role': 'system', 'content': 'Answer in exactly one sentence. Be precise and scientific.'},
        {'role': 'user', 'content': '{{input}}'},
    ]
)

run_experiment(
    experiment_name='concise-scientific-prompt',
    dataset=DATASET_NAME,
    prompt_template=prompt,
    metrics=[
        GalileoMetrics.correctness,
        GalileoMetrics.instruction_adherence,
        GalileoMetrics.ground_truth_adherence,
    ],
    prompt_settings=PromptRunSettings(model_alias='GPT-4o Mini', temperature=0.3, max_tokens=100),
    project=PROJECT_NAME,
)
'Concise prompt experiment complete'


## 5. Run a Code-Based Experiment


In [ ]:
def eli5_runner(row: dict) -> str:
    return f"Imagine you're 5 years old. {row.get('input', '')} works like a simple science trick."

run_experiment(
    experiment_name='eli5-function-experiment',
    dataset=DATASET_NAME,
    function=eli5_runner,
    metrics=[GalileoMetrics.correctness, GalileoMetrics.instruction_adherence],
    project=PROJECT_NAME,
)
'Code-based experiment complete'


## 6. Enable Expression Metrics


In [ ]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[GalileoMetrics.bleu, GalileoMetrics.rouge, GalileoMetrics.input_tone, GalileoMetrics.output_tone],
)


## 7. Enable Confidence Metrics


In [ ]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[GalileoMetrics.prompt_perplexity, GalileoMetrics.uncertainty],
)


## 8. Clean Up the Dataset and Project


In [ ]:
try:
    if dataset_id is not None:
        delete_dataset(id=str(dataset_id))
    else:
        delete_dataset(name=DATASET_NAME, project_name=PROJECT_NAME)
except Exception:
    pass

delete_project(name=PROJECT_NAME)
